In [ ]:
# 1. Imports

from pathlib import Path
import time


# ============================================================
# EXTERNE BIBLIOTHEKEN - Datenverarbeitung & Analyse
# ============================================================
import numpy as np                           # Numerische Berechnungen
import pandas as pd                          # Datenanalyse und TimeSeriesManagement
import ccxt                                  # Kryptobörsen-API (Binance etc.)

from dataclasses import dataclass
import configparser
import hmac
import hashlib
import requests

In [ ]:
# 2. Configuration

# Change run date
RUN_DATE = "2026-03-13"
TOP_N = 100
VS_CURRENCY = "usd"


PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FETCHED_DIR = RAW_DIR / "fetched"
PROCESSED_DIR = DATA_DIR / "processed"

ASSETS_PATH = FETCHED_DIR / "assets.parquet"
ASSET_DAILY_PATH = FETCHED_DIR / "asset_daily"  
ASSET_OHLCV_5M_DIR = FETCHED_DIR / "asset_ohlcv_5m"

SRC_PATH = PROJECT_ROOT / "Src"
SECRET_PATH = SRC_PATH / "secret.cfg"

TARGET_VOLUME_EXCHANGE = "binance"
USD_LIKE_QUOTES = ["USDT", "USDC", "USD"]

OHLCV_INTERVAL = "5m"
BINANCE_KLINE_LIMIT = 1000
OHLCV_START_DATE = RUN_DATE
OHLCV_END_DATE_EXCLUSIVE = "2026-03-14"



OHLCV_5M_COLUMNS = [
    "timestamp",
    "asset_id",
    "exchange",
    "market_symbol",
    "open",
    "high",
    "low",
    "close",
    "volume_base",
    "volume_quote",
    "trade_count",
]

c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\notebooks\01_data_aquisition


In [ ]:
# =========================
# Konfiguration
# =========================
BASE_URL = "https://data-api.binance.vision"
INTERVAL = "5m"          
SLEEP_BETWEEN_REQUESTS = 0.15

@dataclass
class BinanceKeys:
    api_key: str
    secret_key: str

def load_keys() -> BinanceKeys:
    """ Lädt die API-Schlüssel aus der Konfigurationsdatei. """
    cfg = configparser.ConfigParser()
    if not cfg.read(SECRET_PATH):
        raise FileNotFoundError(f"Konnte {SECRET_PATH} nicht lesen.")
    return BinanceKeys(
        api_key=cfg["BINANCE"]["TEST_API_KEY"].strip(),
        secret_key=cfg["BINANCE"]["TEST_SECRET_KEY"].strip()
    )

def _local_ms() -> int:
    return int(time.time() * 1000)

def _ts_ms() -> int:
    # Server synchronisiert: lokale Zeit + Offset
    return _local_ms() + TIME_OFFSET_MS

def _sign(secret_key: str, query_string: str) -> str:
    """ Berechnet die HMAC-SHA256-Signatur für die gegebene Query-String mit dem Secret Key.
    Diese Signatur wird für authentifizierte Anfragen an die Binance-API benötigt."""
    return hmac.new(
        secret_key.encode("utf-8"),
        query_string.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()

def public_get(path: str, params: dict | None = None):
    """ Führt eine GET-Anfrage an die Binance-API durch und gibt die JSON-Antwort zurück."""
    url = f"{TESTNET_REST}{path}"
    r = requests.get(url, params=params, timeout=20)
    if r.status_code >= 400:
        raise RuntimeError(f"Public Fehler {r.status_code}: {r.text}")
    return r.json()

def sync_server_time_offset():
    """
    Holt die Binance-Serverzeit und berechnet den Offset.
    Offset = serverTime - localTime
    """
    global TIME_OFFSET_MS
    try:
        server = public_get("/api/v3/time")
        server_ms = int(server["serverTime"])
        local_ms = _local_ms()
        TIME_OFFSET_MS = server_ms - local_ms
        print(f"[TIME] Offset synchronisiert: {TIME_OFFSET_MS} ms (server-local)")
    except Exception as e:
        print(f"[TIME] Konnte Offset nicht synchronisieren: {e}")

def fetch_ohlcv_for_day() -> pd.DataFrame:
    parquet_files = sorted(ASSET_DAILY_PATH.glob("*.parquet"))
    for file in parquet_files:
        df_month = pd.read_parquet(file)

        df_month["date"] = pd.to_datetime(df_month["date"]).dt.date

        for current_date, df_day in df_month.groupby("date"):
            tickers = df_day["symbol"].dropna().unique().tolist()

            print(f"{current_date}: {len(tickers)} Ticker")

            for symbol in tickers:
                print(f"  Lade {symbol} für {current_date}")
                # hier Binance-Abfrage für genau diesen Tag

TESTNET_REST = "https://testnet.binance.vision"
streams = [f"{ticker.lower()}@kline_{INTERVAL}" for ticker in TICKERS]
stream_path = "/".join(streams)
ws_url = f"wss://stream.testnet.binance.vision/stream?streams={stream_path}"



# globaler Zeit-Offset (Serverzeit - lokale Zeit) in ms
TIME_OFFSET_MS = 0

NameError: name 'dataclass' is not defined

In [ ]:
# ============================================================
# Konfiguration
# ============================================================
BASE_URL = "https://data-api.binance.vision"
INTERVAL = "5m"
SLEEP_BETWEEN_REQUESTS = 0.15

PROJECT_ROOT = Path.cwd()

OUTPUT_PATH = PROJECT_ROOT / "data" / "raw" / "fetched" / "asset_ohlcv_5m"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

OHLCV_COLUMNS = [
    "timestamp",
    "asset_id",
    "exchange",
    "market_symbol",
    "open",
    "high",
    "low",
    "close",
    "volume_base",
    "volume_quote",
    "trade_count",
]


# ============================================================
# Public REST
# ============================================================
SESSION = requests.Session()


def public_get(path: str, params: dict | None = None):
    url = f"{BASE_URL}{path}"
    response = SESSION.get(url, params=params, timeout=20)

    if response.status_code >= 400:
        raise RuntimeError(f"Public Fehler {response.status_code}: {response.text}")

    return response.json()


# ============================================================
# 5m-Klines für genau einen Tag laden
# ============================================================
def fetch_klines_for_day(symbol: str, current_date) -> pd.DataFrame:
    start_dt = pd.Timestamp(current_date)
    end_dt = start_dt + pd.Timedelta(days=1)

    data = public_get(
        "/api/v3/klines",
        {
            "symbol": symbol,
            "interval": INTERVAL,
            "startTime": int(start_dt.timestamp() * 1000),
            "endTime": int(end_dt.timestamp() * 1000),
            "limit": 1000,  # reicht für 5m, da 1 Tag = 288 Kerzen
        },
    )

    if not data:
        return pd.DataFrame(columns=OHLCV_COLUMNS)

    df = pd.DataFrame(
        data,
        columns=[
            "open_time",
            "open",
            "high",
            "low",
            "close",
            "volume_base",
            "close_time",
            "volume_quote",
            "trade_count",
            "taker_buy_base",
            "taker_buy_quote",
            "ignore",
        ],
    )

    df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df["asset_id"] = pd.NA
    df["exchange"] = "binance"
    df["market_symbol"] = symbol

    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "volume_base",
        "volume_quote",
        "trade_count",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[
        [
            "timestamp",
            "asset_id",
            "exchange",
            "market_symbol",
            "open",
            "high",
            "low",
            "close",
            "volume_base",
            "volume_quote",
            "trade_count",
        ]
    ].copy()

    return df.sort_values(["market_symbol", "timestamp"]).reset_index(drop=True)


# ============================================================
# Monatsdatei aus asset_daily verarbeiten
# ============================================================
def process_month_file(file: Path) -> pd.DataFrame:
    df_month = pd.read_parquet(file)
    df_month["date"] = pd.to_datetime(df_month["date"]).dt.date

    month_results = []

    for current_date, df_day in df_month.groupby("date"):
        # falls rank vorhanden ist, sauber sortieren
        if "rank" in df_day.columns:
            df_day = df_day.sort_values("rank")

        tickers = (
            df_day["symbol"]
            .dropna()
            .drop_duplicates()
            .tolist()
        )

        print(f"{file.name} | {current_date}: {len(tickers)} Ticker")

        for symbol in tickers:
            print(f"  Lade {symbol} für {current_date}")

            try:
                df_ohlcv = fetch_klines_for_day(symbol=symbol, current_date=current_date)

                if not df_ohlcv.empty:
                    month_results.append(df_ohlcv)

            except Exception as e:
                print(f"  Fehler bei {symbol} am {current_date}: {e}")

            time.sleep(SLEEP_BETWEEN_REQUESTS)

    if not month_results:
        return pd.DataFrame(columns=OHLCV_COLUMNS)

    df_out = pd.concat(month_results, ignore_index=True)
    df_out = df_out.drop_duplicates(subset=["timestamp", "market_symbol"]).sort_values(
        ["market_symbol", "timestamp"]
    ).reset_index(drop=True)

    return df_out


# ============================================================
# Monatsdatei schreiben oder aktualisieren
# ============================================================
def upsert_month_parquet(output_file: Path, df_new: pd.DataFrame) -> None:
    if df_new.empty:
        print(f"Keine neuen Daten für {output_file.name}")
        return

    if output_file.exists():
        print(f"Aktualisiere bestehende Datei: {output_file.name}")
        df_existing = pd.read_parquet(output_file)

        df_all = pd.concat([df_existing, df_new], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=["timestamp", "market_symbol"])
        df_all = df_all.sort_values(["market_symbol", "timestamp"]).reset_index(drop=True)
    else:
        print(f"Erstelle neue Datei: {output_file.name}")
        df_all = df_new.sort_values(["market_symbol", "timestamp"]).reset_index(drop=True)

    df_all.to_parquet(output_file, index=False)
    print(f"Gespeichert: {output_file}")


# ============================================================
# Hauptschleife
# ============================================================
parquet_files = sorted(ASSET_DAILY_PATH.glob("*.parquet"))

for file in parquet_files:
    print(f"\nVerarbeite: {file.name}")

    df_new_month = process_month_file(file)

    output_file = OUTPUT_PATH / f"{file.stem}_ohlcv_5m.parquet"
    upsert_month_parquet(output_file, df_new_month)


In [30]:
# 4. Helper functions

def ensure_output_dirs() -> None:
    FETCHED_DIR.mkdir(parents=True, exist_ok=True)
    ASSET_OHLCV_5M_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def enforce_columns(df: pd.DataFrame, expected_columns: list[str]) -> pd.DataFrame:
    missing = [c for c in expected_columns if c not in df.columns]
    extra = [c for c in df.columns if c not in expected_columns]

    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if extra:
        raise ValueError(f"Unexpected columns: {extra}")

    return df[expected_columns].copy()


def get_ohlcv_5m_month_path(year: int, month: int) -> Path:
    return ASSET_OHLCV_5M_DIR / f"asset_ohlcv_5m_{year:04d}-{month:02d}.parquet"


def make_binance_exchange():
    exchange = ccxt.binance(
        {
            "enableRateLimit": True,
            "timeout": 60000,
        }
    )
    exchange.load_markets()
    return exchange


def find_binance_usd_like_symbol(exchange, base_symbol: str) -> str | None:
    base_symbol = str(base_symbol).upper().strip()

    for quote in USD_LIKE_QUOTES:
        candidate = f"{base_symbol}/{quote}"
        if candidate in exchange.markets:
            market = exchange.markets[candidate]
            if market.get("spot") and market.get("active", True):
                return candidate

    return None


def fetch_binance_quote_volume_for_date(exchange, market_symbol: str, run_date: str) -> float | None:
    market = exchange.market(market_symbol)

    start_ts = exchange.parse8601(f"{run_date}T00:00:00Z")
    end_ts = start_ts + 24 * 60 * 60 * 1000

    rows = exchange.publicGetKlines(
        {
            "symbol": market["id"],
            "interval": "1d",
            "startTime": start_ts,
            "endTime": end_ts,
            "limit": 1,
        }
    )

    if not rows:
        return None

    row = rows[0]

    candle_date = pd.to_datetime(int(row[0]), unit="ms", utc=True).strftime("%Y-%m-%d")
    if candle_date != run_date:
        return None

    return float(row[7])

In [41]:
# 14. OHLCV 5m configuration

OHLCV_5M_PATH = PROCESSED_DIR / "asset_ohlcv_5m.parquet"

OHLCV_5M_COLUMNS = [
    "timestamp",
    "asset_id",
    "exchange",
    "market_symbol",
    "open",
    "high",
    "low",
    "close",
    "volume_base",
    "volume_quote",
    "trade_count",
]

OHLCV_INTERVAL = "5m"
BINANCE_KLINE_LIMIT = 1000

OHLCV_START_DATE = RUN_DATE
OHLCV_END_DATE_EXCLUSIVE = "2026-03-14"

In [42]:
# 15. OHLCV 5m helper functions

def fetch_binance_klines_raw(
    exchange,
    market_symbol: str,
    interval: str,
    start_ts: int,
    end_ts: int,
    limit: int = 1000,
    max_retries: int = 5,
):
    market = exchange.market(market_symbol)
    last_error = None

    for attempt in range(max_retries):
        try:
            rows = exchange.publicGetKlines(
                {
                    "symbol": market["id"],
                    "interval": interval,
                    "startTime": start_ts,
                    "endTime": end_ts,
                    "limit": limit,
                }
            )
            time.sleep(exchange.rateLimit / 1000)
            return rows

        except (ccxt.RequestTimeout, ccxt.NetworkError) as e:
            last_error = e
            wait_seconds = min(60, 5 * (attempt + 1))
            print(
                f"Binance timeout/network error for {market_symbol} "
                f"| attempt {attempt + 1}/{max_retries} "
                f"| sleeping {wait_seconds}s"
            )
            time.sleep(wait_seconds)

    raise last_error or RuntimeError(f"Failed to fetch klines for {market_symbol}")


def collect_binance_ohlcv_5m_for_range(
    exchange,
    asset_id: int,
    market_symbol: str,
    start_date: str,
    end_date: str,
) -> list[dict]:
    start_ts = exchange.parse8601(f"{start_date}T00:00:00Z")
    end_ts = exchange.parse8601(f"{end_date}T00:00:00Z")

    if start_ts is None or end_ts is None:
        raise ValueError("Could not parse start_date or end_date.")

    all_rows = []
    current_ts = start_ts

    while current_ts < end_ts:
        rows = fetch_binance_klines_raw(
            exchange=exchange,
            market_symbol=market_symbol,
            interval=OHLCV_INTERVAL,
            start_ts=current_ts,
            end_ts=end_ts,
            limit=BINANCE_KLINE_LIMIT,
            max_retries=5,
        )

        if not rows:
            break

        parsed_rows = []
        for row in rows:
            parsed_rows.append(
                {
                    "timestamp": pd.to_datetime(int(row[0]), unit="ms", utc=True),
                    "asset_id": asset_id,
                    "exchange": "binance",
                    "market_symbol": market_symbol,
                    "open": float(row[1]),
                    "high": float(row[2]),
                    "low": float(row[3]),
                    "close": float(row[4]),
                    "volume_base": float(row[5]),
                    "volume_quote": float(row[7]),
                    "trade_count": int(row[8]),
                }
            )

        all_rows.extend(parsed_rows)

        last_open_time = int(rows[-1][0])
        next_ts = last_open_time + 5 * 60 * 1000

        if next_ts <= current_ts:
            break

        current_ts = next_ts

    return all_rows

In [43]:
# 16. Match Binance market symbols for all assets

asset_binance_markets_df = asset_df[["asset_id", "symbol"]].copy()

asset_binance_markets_df["market_symbol"] = asset_binance_markets_df["symbol"].apply(
    lambda s: find_binance_usd_like_symbol(binance, s)
)

asset_binance_markets_df

,asset_id,symbol,market_symbol
0,1,BTC,BTC/USDT
1,2,ETH,ETH/USDT
2,3,USDT,USDT/USD
3,4,XRP,XRP/USDT
4,5,BNB,BNB/USDT
...,...,...,...
95,96,FET,FET/USDT
96,97,VIRTUAL,VIRTUAL/USDT
97,98,PENGU,PENGU/USDT
98,99,CAKE,CAKE/USDT


In [44]:
# 17. Fetch 5-minute OHLCV data for all matched Binance markets

ohlcv_5m_rows = []
ohlcv_5m_errors = []

for _, row in asset_binance_markets_df.iterrows():
    asset_id = int(row["asset_id"])
    symbol = row["symbol"]
    market_symbol = row["market_symbol"]

    if market_symbol is None:
        ohlcv_5m_errors.append(
            {
                "asset_id": asset_id,
                "symbol": symbol,
                "market_symbol": None,
                "error_message": "No Binance USD-like market found.",
            }
        )
        continue

    try:
        rows = collect_binance_ohlcv_5m_for_range(
            exchange=binance,
            asset_id=asset_id,
            market_symbol=market_symbol,
            start_date=OHLCV_START_DATE,
            end_date=OHLCV_END_DATE_EXCLUSIVE,
        )
        ohlcv_5m_rows.extend(rows)
        print(f"Fetched {len(rows)} rows for {symbol} ({market_symbol})")
    except Exception as e:
        ohlcv_5m_errors.append(
            {
                "asset_id": asset_id,
                "symbol": symbol,
                "market_symbol": market_symbol,
                "error_message": repr(e),
            }
        )
        print(f"OHLCV fetch failed for {symbol} ({market_symbol}): {e}")

Fetched 289 rows for BTC (BTC/USDT)
Fetched 289 rows for ETH (ETH/USDT)
Fetched 289 rows for USDT (USDT/USD)
Fetched 289 rows for XRP (XRP/USDT)
Fetched 289 rows for BNB (BNB/USDT)
Fetched 289 rows for USDC (USDC/USDT)
Fetched 289 rows for SOL (SOL/USDT)
Fetched 289 rows for TRX (TRX/USDT)
OHLCV fetch failed for FIGR_HELOC (nan): can only concatenate str (not "float") to str
Fetched 289 rows for DOGE (DOGE/USDT)
OHLCV fetch failed for WBT (nan): can only concatenate str (not "float") to str
OHLCV fetch failed for USDS (nan): can only concatenate str (not "float") to str
Fetched 289 rows for ADA (ADA/USDT)
OHLCV fetch failed for HYPE (nan): can only concatenate str (not "float") to str
Fetched 289 rows for BCH (BCH/USDT)
OHLCV fetch failed for LEO (nan): can only concatenate str (not "float") to str
Fetched 289 rows for LINK (LINK/USDT)
OHLCV fetch failed for XMR (nan): can only concatenate str (not "float") to str
Fetched 289 rows for USDE (USDE/USDT)
OHLCV fetch failed for CC (nan): c

In [45]:
# 18. Build asset_ohlcv_5m dataframe

asset_ohlcv_5m_df = pd.DataFrame(ohlcv_5m_rows)

if asset_ohlcv_5m_df.empty:
    asset_ohlcv_5m_df = pd.DataFrame(columns=OHLCV_5M_COLUMNS)
else:
    asset_ohlcv_5m_df = enforce_columns(asset_ohlcv_5m_df, OHLCV_5M_COLUMNS)
    asset_ohlcv_5m_df = asset_ohlcv_5m_df.sort_values(
        ["asset_id", "timestamp"]
    ).reset_index(drop=True)

asset_ohlcv_5m_df

,timestamp,asset_id,exchange,market_symbol,open,high,low,close,volume_base,volume_quote,trade_count
0,2026-03-13 00:00:00+00:00,1,binance,BTC/USDT,70541.3400,70541.3400,70386.0100,70480.5400,177.17450,1.249181e+07,16395
1,2026-03-13 00:05:00+00:00,1,binance,BTC/USDT,70480.5400,70633.7900,70480.5400,70633.7800,47.60617,3.360137e+06,16210
2,2026-03-13 00:10:00+00:00,1,binance,BTC/USDT,70633.7900,71118.5900,70633.7800,71118.5800,438.21589,3.108579e+07,45915
3,2026-03-13 00:15:00+00:00,1,binance,BTC/USDT,71118.5800,71680.0000,71105.3300,71270.1800,594.63126,4.246651e+07,76327
4,2026-03-13 00:20:00+00:00,1,binance,BTC/USDT,71270.1800,71759.6000,71260.4200,71530.0000,367.08836,2.628977e+07,63933
...,...,...,...,...,...,...,...,...,...,...,...
17335,2026-03-13 23:40:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,1557.00000,1.556066e+03,16
17336,2026-03-13 23:45:00+00:00,100,binance,TUSD/USDT,0.9995,0.9995,0.9994,0.9994,69.00000,6.895980e+01,7
17337,2026-03-13 23:50:00+00:00,100,binance,TUSD/USDT,0.9994,0.9995,0.9994,0.9995,152.00000,1.519197e+02,7
17338,2026-03-13 23:55:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,40.00000,3.997600e+01,5


In [46]:
# 19. Build OHLCV 5m error dataframe

ohlcv_5m_error_df = pd.DataFrame(ohlcv_5m_errors)
ohlcv_5m_error_df

,asset_id,symbol,market_symbol,error_message
0,9,FIGR_HELOC,NaN,"TypeError('can only concatenate str (not ""floa..."
1,11,WBT,NaN,"TypeError('can only concatenate str (not ""floa..."
2,12,USDS,NaN,"TypeError('can only concatenate str (not ""floa..."
3,14,HYPE,NaN,"TypeError('can only concatenate str (not ""floa..."
4,16,LEO,NaN,"TypeError('can only concatenate str (not ""floa..."
5,18,XMR,NaN,"TypeError('can only concatenate str (not ""floa..."
6,20,CC,NaN,"TypeError('can only concatenate str (not ""floa..."
7,27,RAIN,NaN,"TypeError('can only concatenate str (not ""floa..."
8,28,DAI,NaN,"TypeError('can only concatenate str (not ""floa..."
9,30,PYUSD,NaN,"TypeError('can only concatenate str (not ""floa..."


In [47]:
# 20. Validate asset_ohlcv_5m dataframe

if asset_ohlcv_5m_df.duplicated(subset=["timestamp", "asset_id", "exchange", "market_symbol"]).any():
    raise ValueError("Duplicate OHLCV rows found.")

for col in ["open", "high", "low", "close", "volume_base", "volume_quote"]:
    if (pd.to_numeric(asset_ohlcv_5m_df[col], errors="coerce").dropna() < 0).any():
        raise ValueError(f"Negative values found in {col}.")

if (pd.to_numeric(asset_ohlcv_5m_df["trade_count"], errors="coerce").dropna() < 0).any():
    raise ValueError("Negative trade_count found.")

print("asset_ohlcv_5m validation checks passed.")

asset_ohlcv_5m validation checks passed.


In [48]:
# 21. Save asset_ohlcv_5m parquet files by month

ensure_output_dirs()

if asset_ohlcv_5m_df.empty:
    print("asset_ohlcv_5m_df is empty. No monthly parquet files were written.")
else:
    asset_ohlcv_5m_df = asset_ohlcv_5m_df.copy()
    asset_ohlcv_5m_df["year"] = asset_ohlcv_5m_df["timestamp"].dt.year
    asset_ohlcv_5m_df["month"] = asset_ohlcv_5m_df["timestamp"].dt.month

    written_paths = []

    for (year, month), month_df in asset_ohlcv_5m_df.groupby(["year", "month"], sort=True):
        output_path = get_ohlcv_5m_month_path(year=int(year), month=int(month))
        month_df = month_df.drop(columns=["year", "month"]).copy()
        month_df.to_parquet(output_path, index=False)
        written_paths.append(output_path)

    print("Saved monthly OHLCV 5m files:")
    for path in written_paths:
        print(path)

Saved monthly OHLCV 5m files:
../../data/raw/fetched/asset_ohlcv_5m/asset_ohlcv_5m_2026-03.parquet


In [49]:
# 22. OHLCV 5m summary and preview

print("OHLCV 5m summary")
print(f"OHLCV_START_DATE: {OHLCV_START_DATE}")
print(f"OHLCV_END_DATE_EXCLUSIVE: {OHLCV_END_DATE_EXCLUSIVE}")
print(f"rows: {len(asset_ohlcv_5m_df)}")
print(f"assets with errors: {len(ohlcv_5m_error_df)}")
print(f"OHLCV directory: {ASSET_OHLCV_5M_DIR}")

display(asset_ohlcv_5m_df)
display(ohlcv_5m_error_df)

OHLCV 5m summary
OHLCV_START_DATE: 2026-03-13
OHLCV_END_DATE_EXCLUSIVE: 2026-03-14
rows: 17340
assets with errors: 40
OHLCV directory: ../../data/raw/fetched/asset_ohlcv_5m


,timestamp,asset_id,exchange,market_symbol,open,high,low,close,volume_base,volume_quote,trade_count,year,month
0,2026-03-13 00:00:00+00:00,1,binance,BTC/USDT,70541.3400,70541.3400,70386.0100,70480.5400,177.17450,1.249181e+07,16395,2026,3
1,2026-03-13 00:05:00+00:00,1,binance,BTC/USDT,70480.5400,70633.7900,70480.5400,70633.7800,47.60617,3.360137e+06,16210,2026,3
2,2026-03-13 00:10:00+00:00,1,binance,BTC/USDT,70633.7900,71118.5900,70633.7800,71118.5800,438.21589,3.108579e+07,45915,2026,3
3,2026-03-13 00:15:00+00:00,1,binance,BTC/USDT,71118.5800,71680.0000,71105.3300,71270.1800,594.63126,4.246651e+07,76327,2026,3
4,2026-03-13 00:20:00+00:00,1,binance,BTC/USDT,71270.1800,71759.6000,71260.4200,71530.0000,367.08836,2.628977e+07,63933,2026,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17335,2026-03-13 23:40:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,1557.00000,1.556066e+03,16,2026,3
17336,2026-03-13 23:45:00+00:00,100,binance,TUSD/USDT,0.9995,0.9995,0.9994,0.9994,69.00000,6.895980e+01,7,2026,3
17337,2026-03-13 23:50:00+00:00,100,binance,TUSD/USDT,0.9994,0.9995,0.9994,0.9995,152.00000,1.519197e+02,7,2026,3
17338,2026-03-13 23:55:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,40.00000,3.997600e+01,5,2026,3


,asset_id,symbol,market_symbol,error_message
0,9,FIGR_HELOC,NaN,"TypeError('can only concatenate str (not ""floa..."
1,11,WBT,NaN,"TypeError('can only concatenate str (not ""floa..."
2,12,USDS,NaN,"TypeError('can only concatenate str (not ""floa..."
3,14,HYPE,NaN,"TypeError('can only concatenate str (not ""floa..."
4,16,LEO,NaN,"TypeError('can only concatenate str (not ""floa..."
5,18,XMR,NaN,"TypeError('can only concatenate str (not ""floa..."
6,20,CC,NaN,"TypeError('can only concatenate str (not ""floa..."
7,27,RAIN,NaN,"TypeError('can only concatenate str (not ""floa..."
8,28,DAI,NaN,"TypeError('can only concatenate str (not ""floa..."
9,30,PYUSD,NaN,"TypeError('can only concatenate str (not ""floa..."
